# PVGIS Pipeline

Single driver notebook for the two supported training modes: `pvgis_only` and `real_plants_pvgis`. The notebook owns run parameters and W&B logging configuration; Python modules own data loading, model code, training, and evaluation.

## 1. Setup

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
os.environ['PYTHONPATH'] = str(REPO_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
PYTHON = sys.executable

def run(cmd, env=None):
    cmd = [str(c) for c in cmd]
    print('$ ' + ' '.join(cmd))
    return subprocess.run(cmd, cwd=REPO_ROOT, env=env, check=True)

def csv(values):
    return ','.join(str(v) for v in values)

print('repo root:', REPO_ROOT)
print('python   :', PYTHON)

## 2. Configuration

In [ ]:
PIPELINE_MODE = 'pvgis_only'  # 'pvgis_only' or 'real_plants_pvgis'

PVGIS_DIR = Path('/data/SentinelPV/pvgis_data/data/pvgis_summed_irradiance')
PVGIS_FILE_TEMPLATE = 'piedmont_pvgis_{year}.nc'
TRAIN_YEARS = [2016, 2017, 2018]
TEST_YEAR = 2019
TARGET_VARIABLE = 'pv_power_output'
FEATURE_SET = 'full'

RUN_ANOMALY_LABELING = True
CLIMATOLOGY_START_YEAR = 2005
CLIMATOLOGY_END_YEAR = 2023
ANOMALY_QUANTILE = 0.975
ANOMALY_WINDOW_DAYS = 15
ANOMALY_OUT_DIR = Path(f'outputs/pvgis_anomaly_{TEST_YEAR}_{CLIMATOLOGY_START_YEAR}_{CLIMATOLOGY_END_YEAR}_w{ANOMALY_WINDOW_DAYS}_q{str(ANOMALY_QUANTILE).replace(".", "")}')
ANOMALY_SCORES = ANOMALY_OUT_DIR / 'pvgis_climatology_scores.csv'
USE_ANOMALY_SCORES = True

DATA_YEAR = 2019
SENTINEL_DIR = Path('/data/SentinelPV/energy_data/piemonte_energy_data/single_ups')
PLANT_MAPPING_PATH = Path('data/plant_mapping.csv')
ENERGY_COORDS_PATH = Path('data/energy_with_coordinates.csv')
REAL_PVGIS_PATH = Path(f'data/piedmont_pvgis_{DATA_YEAR}.nc')
REAL_FEATURE_SET = 'real_plants_pvgis'

SEEDS = [42]
SEQ_LEN = 24
PATCH_LEN = 4
STRIDE = 2
HORIZON = 1
N_EPOCHS = 10
BATCH_SIZE = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.2
MAX_TRAIN_SAMPLES = None
MAX_TEST_SAMPLES = None
MAX_STEPS_PER_EPOCH = None
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 1e-4
BILSTM_POOLING = 'attn'

PEAK_ALPHA = 2.5
PEAK_GAMMA = 2.0
PEAK_LOSS_WEIGHT = 0.25
UNDER_PENALTY = 3.0
ETA_MAX = 0.98
CALIBRATION_KPI = 'none'
QS_LOSS_WEIGHTING = False
QS_LOSS_FLOOR = 0.2
APPLY_OUTLIER_FILTER = False
OUTLIER_QS_DAYTIME_THRESHOLD = 0.30
OUTLIER_MIN_VALID_DAYTIME = 200

MC_DROPOUT = False
MC_SAMPLES = 30
ENABLE_POSTHOC_CALIBRATION = False
CALIBRATION_YEARS = []
COVERAGE_TARGET = 0.95
CALIBRATION_STRATEGY = 'global'
CALIBRATION_ANOMALY_SCORES = None
SAVE_ENSEMBLE_PREDICTIONS = False
ENSEMBLE_ID = None

WANDB_MODE = 'online'
WANDB_PROJECT = 'PhysiQ-PV'
WANDB_ENTITY = 'albertopedalino-politecnico-di-torino'
ENABLE_WANDB = True
WANDB_LOG_PREDICTIONS = False

PVGIS_RUN_NAME = f'pvgis_only_stgnn_{FEATURE_SET}_seq{SEQ_LEN}_seed{SEEDS[0]}'
REAL_RUN_NAME = (
    '{mode}_{feature_set}_seq{seq_len}_peakw{peak_loss_weight:g}'
    '_pool{bilstm_pooling}{quality_suffix}_seed{seed}'
)
CHECKPOINT_DIR_BASE = f'checkpoints/real_pvgis_seq{SEQ_LEN}'
PVGIS_OUT_DIR = 'outputs/wandb_pvgis_stgnn/{wandb_run_id}' if ENABLE_WANDB else f'outputs/pvgis_stgnn_{TEST_YEAR}_{FEATURE_SET}'
REAL_WANDB_TAGS = ['real-plants-pvgis', 'st-gnn', f'seq_len_{SEQ_LEN}']

RUN_PIPELINE = True

env = dict(os.environ)
env.update({
    'WANDB_MODE': WANDB_MODE,
    'DATA_YEAR': str(DATA_YEAR),
    'SENTINEL_DIR': str(SENTINEL_DIR),
    'PLANT_MAPPING_PATH': str(PLANT_MAPPING_PATH),
    'ENERGY_COORDS_PATH': str(ENERGY_COORDS_PATH),
    'PVGIS_PATH': str(REAL_PVGIS_PATH),
    'REAL_FEATURE_SET': REAL_FEATURE_SET,
    'SEEDS': csv(SEEDS),
    'SEQ_LEN': str(SEQ_LEN),
    'PATCH_LEN': str(PATCH_LEN),
    'STRIDE': str(STRIDE),
    'N_EPOCHS': str(N_EPOCHS),
    'BATCH_SIZE': str(BATCH_SIZE),
    'LR': str(LR),
    'WEIGHT_DECAY': str(WEIGHT_DECAY),
    'DROPOUT': str(DROPOUT),
    'MAX_STEPS_PER_EPOCH': '' if MAX_STEPS_PER_EPOCH is None else str(MAX_STEPS_PER_EPOCH),
    'EARLY_STOPPING_PATIENCE': str(EARLY_STOPPING_PATIENCE),
    'EARLY_STOPPING_MIN_DELTA': str(EARLY_STOPPING_MIN_DELTA),
    'BILSTM_POOLING': BILSTM_POOLING,
    'PEAK_ALPHA': str(PEAK_ALPHA),
    'PEAK_GAMMA': str(PEAK_GAMMA),
    'PEAK_LOSS_WEIGHT': str(PEAK_LOSS_WEIGHT),
    'UNDER_PENALTY': str(UNDER_PENALTY),
    'ETA_MAX': str(ETA_MAX),
    'CALIBRATION_KPI': CALIBRATION_KPI,
    'QS_LOSS_WEIGHTING': str(int(QS_LOSS_WEIGHTING)),
    'QS_LOSS_FLOOR': str(QS_LOSS_FLOOR),
    'APPLY_OUTLIER_FILTER': str(int(APPLY_OUTLIER_FILTER)),
    'OUTLIER_QS_DAYTIME_THRESHOLD': str(OUTLIER_QS_DAYTIME_THRESHOLD),
    'OUTLIER_MIN_VALID_DAYTIME': str(OUTLIER_MIN_VALID_DAYTIME),
    'CHECKPOINT_DIR_BASE': CHECKPOINT_DIR_BASE,
    'WANDB_PROJECT': WANDB_PROJECT,
    'WANDB_ENTITY': WANDB_ENTITY,
    'USE_WANDB': str(int(ENABLE_WANDB)),
    'WANDB_RUN_NAME': REAL_RUN_NAME,
    'WANDB_TAGS': ','.join(REAL_WANDB_TAGS),
})

print(json.dumps({
    'PIPELINE_MODE': PIPELINE_MODE,
    'PVGIS_DIR': str(PVGIS_DIR),
    'TRAIN_YEARS': TRAIN_YEARS,
    'TEST_YEAR': TEST_YEAR,
    'FEATURE_SET': FEATURE_SET,
    'SEQ_LEN': SEQ_LEN,
    'N_EPOCHS': N_EPOCHS,
    'BATCH_SIZE': BATCH_SIZE,
    'LR': LR,
    'DROPOUT': DROPOUT,
    'MC_DROPOUT': MC_DROPOUT,
    'WANDB_MODE': WANDB_MODE,
    'WANDB_PROJECT': WANDB_PROJECT,
    'WANDB_ENTITY': WANDB_ENTITY,
    'PVGIS_RUN_NAME': PVGIS_RUN_NAME,
    'REAL_RUN_NAME': REAL_RUN_NAME,
}, indent=2))

## 3. Input Checks

In [ ]:
if PIPELINE_MODE not in {'pvgis_only', 'real_plants_pvgis'}:
    raise ValueError("PIPELINE_MODE must be 'pvgis_only' or 'real_plants_pvgis'")

checks = {}
if PIPELINE_MODE == 'pvgis_only':
    checks['PVGIS dir'] = PVGIS_DIR.is_dir()
    checks['test PVGIS file'] = (PVGIS_DIR / PVGIS_FILE_TEMPLATE.format(year=TEST_YEAR)).exists()
    for year in TRAIN_YEARS:
        checks[f'train PVGIS file {year}'] = (PVGIS_DIR / PVGIS_FILE_TEMPLATE.format(year=year)).exists()
else:
    checks['Sentinel dir'] = SENTINEL_DIR.is_dir()
    checks['plant mapping'] = PLANT_MAPPING_PATH.exists()
    checks['energy coords'] = ENERGY_COORDS_PATH.exists()
    checks['real PVGIS file'] = REAL_PVGIS_PATH.exists()

for name, ok in checks.items():
    print(('OK     ' if ok else 'MISSING') + '  ' + name)

missing = [name for name, ok in checks.items() if not ok]
if missing:
    raise FileNotFoundError('Missing required inputs: ' + ', '.join(missing))

## 4. Optional PVGIS Anomaly Labels

In [ ]:
if PIPELINE_MODE == 'pvgis_only' and RUN_ANOMALY_LABELING and not ANOMALY_SCORES.exists():
    run([
        PYTHON, 'scripts/run_pvgis_climatology_anomaly.py',
        '--year', TEST_YEAR,
        '--pvgis-path', PVGIS_DIR / PVGIS_FILE_TEMPLATE.format(year=TEST_YEAR),
        '--pvgis-climatology-dir', PVGIS_DIR,
        '--climatology-start-year', CLIMATOLOGY_START_YEAR,
        '--climatology-end-year', CLIMATOLOGY_END_YEAR,
        '--quantile', ANOMALY_QUANTILE,
        '--climatology-window-days', ANOMALY_WINDOW_DAYS,
        '--out-dir', ANOMALY_OUT_DIR,
        '--file-template', PVGIS_FILE_TEMPLATE,
    ], env=env)
else:
    print('Anomaly labeling skipped.')

if PIPELINE_MODE == 'pvgis_only' and USE_ANOMALY_SCORES:
    print('anomaly scores:', ANOMALY_SCORES if ANOMALY_SCORES.exists() else '(not available)')

## 5. W&B

In [ ]:
try:
    import wandb
    print('wandb version:', wandb.__version__)
except ImportError:
    print('wandb is not installed in this environment')
print('enabled:', ENABLE_WANDB)
print('mode   :', WANDB_MODE)
print('project:', WANDB_PROJECT)
print('entity :', WANDB_ENTITY)

## 6. Build Command

In [ ]:
def build_pvgis_only_cmd():
    cmd = [
        PYTHON, 'main.py', '--mode', 'pvgis_stgnn',
        '--pvgis-dir', PVGIS_DIR,
        '--train-years', csv(TRAIN_YEARS),
        '--test-year', TEST_YEAR,
        '--file-template', PVGIS_FILE_TEMPLATE,
        '--target-variable', TARGET_VARIABLE,
        '--feature-set', FEATURE_SET,
        '--seq-len', SEQ_LEN,
        '--horizon', HORIZON,
        '--epochs', N_EPOCHS,
        '--batch-size', BATCH_SIZE,
        '--lr', LR,
        '--dropout', DROPOUT,
        '--out-dir', PVGIS_OUT_DIR,
    ]
    if MAX_TRAIN_SAMPLES is not None:
        cmd += ['--max-train-samples', MAX_TRAIN_SAMPLES]
    if MAX_TEST_SAMPLES is not None:
        cmd += ['--max-test-samples', MAX_TEST_SAMPLES]
    if USE_ANOMALY_SCORES and ANOMALY_SCORES.exists():
        cmd += ['--anomaly-scores', ANOMALY_SCORES]
    if MC_DROPOUT:
        cmd += ['--mc-dropout', '--mc-samples', MC_SAMPLES, '--coverage-target', COVERAGE_TARGET]
    if ENABLE_POSTHOC_CALIBRATION:
        cmd += ['--enable-posthoc-calibration']
        if CALIBRATION_YEARS:
            cmd += ['--calibration-years', csv(CALIBRATION_YEARS)]
        cmd += ['--calibration-strategy', CALIBRATION_STRATEGY]
        if CALIBRATION_ANOMALY_SCORES is not None:
            cmd += ['--calibration-anomaly-scores', CALIBRATION_ANOMALY_SCORES]
    if SAVE_ENSEMBLE_PREDICTIONS:
        cmd += ['--save-ensemble-predictions']
        if ENSEMBLE_ID:
            cmd += ['--ensemble-id', ENSEMBLE_ID]
    if ENABLE_WANDB:
        cmd += ['--wandb', '--wandb-project', WANDB_PROJECT, '--wandb-run-name', PVGIS_RUN_NAME]
        if WANDB_ENTITY:
            cmd += ['--wandb-entity', WANDB_ENTITY]
        if WANDB_LOG_PREDICTIONS:
            cmd += ['--wandb-log-predictions']
    return cmd

def build_real_plants_pvgis_cmd():
    return [PYTHON, 'main.py', '--mode', 'real_plants_pvgis']

cmd = build_pvgis_only_cmd() if PIPELINE_MODE == 'pvgis_only' else build_real_plants_pvgis_cmd()
print('$ ' + ' '.join(str(c) for c in cmd))

## 7. Run Pipeline

In [ ]:
if RUN_PIPELINE:
    run(cmd, env=env)
else:
    print('Pipeline skipped. Set RUN_PIPELINE=True to execute.')

## 8. Outputs

In [ ]:
if PIPELINE_MODE == 'pvgis_only':
    root = Path('outputs/wandb_pvgis_stgnn')
    paths = sorted(root.glob('*')) if root.exists() else []
    print('pvgis output dirs:', len(paths))
    for path in paths[-10:]:
        print(path)
else:
    checkpoint_dirs = sorted(Path('checkpoints').glob(f'{Path(CHECKPOINT_DIR_BASE).name}_pool*seed*'))
    print('checkpoint dirs:', len(checkpoint_dirs))
    for path in checkpoint_dirs:
        summary = path / 'loss_history.json'
        if summary.exists():
            data = json.loads(summary.read_text())
            best = min(data.get('val', [float('nan')]))
            print(f'{path}  best_val={best:.4f}')
        else:
            print(path)